### Task 1

In [2]:
import math
import pandas as pd

insurance_df = pd.read_csv('Data/Insurance.csv')
insurance_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


1. 7 Columns, 1338 rows
2. Each column represents a quality about a person. Each row is a specific person.
3. The label column is `charges`
4. The rest of the columns are features: age, sex, bmi, children, smoker, region

### Task 2

In [3]:
brian_guess = 3000
jesse_guess = 20000

### Task 3

In [4]:
mean_baseline = round(insurance_df['charges'].mean(), 2)
print(f'Mean Baseline: ${mean_baseline}')
insurance_df['charge_baseline'] = mean_baseline

# Our guesses were not very close. Almost the same distance, one higher and one lower

Mean Baseline: $13270.42


### Task 4

In [5]:
insurance_df['baseline_error'] = abs(insurance_df['charges'] - insurance_df['charge_baseline'])
baseline_mae = round(insurance_df['baseline_error'].mean(),2)
print(f'Baseline MAE: ${baseline_mae}')

# Our baseline MAE is $9000 which means that this very basic predictive model is generally accurate
#      within $9000.

Baseline MAE: $9091.13


In [6]:
insurance_df.head()

,age,sex,bmi,children,smoker,region,charges,charge_baseline,baseline_error
0,19,female,27.900,0,yes,southwest,16884.92400,13270.42,3614.50400
1,18,male,33.770,1,no,southeast,1725.55230,13270.42,11544.86770
2,28,male,33.000,3,no,southeast,4449.46200,13270.42,8820.95800
3,33,male,22.705,0,no,northwest,21984.47061,13270.42,8714.05061
4,32,male,28.880,0,no,northwest,3866.85520,13270.42,9403.56480


### Task 5

In [7]:
baseline_mse = insurance_df['baseline_error'].apply(lambda x: x**2).mean()
print(f'Baseline MSE: {baseline_mse}')

baseline_rmse = round(math.sqrt(baseline_mse),2)
print(f'Baseline RMSE: ${baseline_rmse}')


# Units for MAE and RMSE are Dollars (dolla dolla bills yall)
# RMSE is prefered when larger errors are more damaging, so we penalize them.

Baseline MSE: 146542766.49355304
Baseline RMSE: $12105.48


### Task 6

In [8]:
smoker_mean = insurance_df[insurance_df["smoker"] == 'yes']['charges'].mean()
non_smoker_mean = insurance_df[insurance_df["smoker"] == 'no']['charges'].mean()

insurance_df["smoker_mean"] = insurance_df["smoker"].map({'yes': smoker_mean, 'no': non_smoker_mean})

error = [abs(c - a) for c, a in zip(insurance_df["charges"], insurance_df["smoker_mean"])]

grouped_mae = sum(error) / len(error)

print(grouped_mae)

5662.08960934306


#Task 7

In [9]:
x = insurance_df.drop(columns=["charges", "charge_baseline", "baseline_error", "smoker_mean"])
y = insurance_df["charges"]

#So we can test how close we get to y by predicting the charges with x

#Task 8

In [18]:
X_Train = x.iloc[:int(len(x) * .8)]
Y_Train = y.iloc[:int(len(x) * .8)]
X_test = x.iloc[int(len(x) * .8):]
Y_test = y.iloc[int(len(x) * .8):]

print("X_Train rows:", len(X_Train))
print("Y_train rows:", len(Y_Train))
print("X_test rows:", len(X_test))
print("Y_test rows:", len(Y_test))

print(Y_Train)

#It can memorize the specific data too well. We won't know if it's giving good predictions or it just memorized the data.
#Overfitting

X_Train rows: 1070
Y_train rows: 1070
X_test rows: 268
Y_test rows: 268
0       16884.92400
1        1725.55230
2        4449.46200
3       21984.47061
4        3866.85520
           ...     
1065     7045.49900
1066     8978.18510
1067     5757.41345
1068    14349.85440
1069    10928.84900
Name: charges, Length: 1070, dtype: float64


#Task 9

In [36]:
mean_baseline = round(Y_Train.mean(), 2)
test_baseline_mae = round(abs(Y_test - mean_baseline).mean(),2)
smoker = X_Train.merge(Y_Train, left_on=X_Train.index, right_on=Y_Train.index)
smoker_mean = round(smoker[smoker["smoker"] == "yes"]["charges"].mean(), 2)
non_smoker_mean = round(smoker[smoker["smoker"] == "no"]["charges"].mean(), 2)


def get_group_mae(df):
    if df["smoker"] == "yes":
       return (abs(df["charges"] - smoker_mean))
    else:
        return (abs(df["charges"] - non_smoker_mean))

test_group_mae = smoker.apply(get_group_mae).mean()


KeyError: 'smoker'

0        True
1       False
2       False
3       False
4       False
        ...  
1065    False
1066    False
1067    False
1068    False
1069    False
Name: smoker, Length: 1070, dtype: bool